#Pandas Data Transformation

##Topics covered in this notebook

**1. Creating new columns**

**2. Column arithmetic operations**

**3. apply() function**

**4. map() function**

**5. lambda functions**

**6. Renaming columns**

**7. Dropping columns**

**8. assign() method**

**9. Replacing values using replace()**

**10. Conditional column creation**





In [99]:
import pandas as pd

data = {
    "employee_id":[101,102,103,104,105],
    "name":["Ali Khan","Sara Ahmed","Omar Malik","Ayesha Noor","Bilal Raza"],
    "city":["London","Paris","London","Berlin","Paris"],
    "department":["Sales","IT","Finance","Sales","IT"],
    "salary":[5000,6500,6000,5200,7000],
    "bonus":[500,700,600,450,800],
    "experience":[3,5,4,2,6]
}

df = pd.DataFrame(data)
df

,employee_id,name,city,department,salary,bonus,experience
0,101,Ali Khan,London,Sales,5000,500,3
1,102,Sara Ahmed,Paris,IT,6500,700,5
2,103,Omar Malik,London,Finance,6000,600,4
3,104,Ayesha Noor,Berlin,Sales,5200,450,2
4,105,Bilal Raza,Paris,IT,7000,800,6


#1. Creating New Columns in Pandas
#And
#2. Column Arithmetic Operations

###We create a new column that calculates total compensation of employees.

In [100]:
df['total_salary'] = df['salary'] + df['bonus']
df['total_salary']

,total_salary
0,5500
1,7200
2,6600
3,5650
4,7800


####Here Pandas performs a vectorized operation, meaning the calculation is applied to the entire column automatically.

####New columns can also be created using subtraction.

####For example, suppose the company deducts 10% tax from salary.

In [101]:
df['salary_after_tax']= df['salary'] - (df['salary']*0.10)
df['salary_after_tax']

,salary_after_tax
0,4500.0
1,5850.0
2,5400.0
3,4680.0
4,6300.0


####We can also create columns using multiplication.

####Suppose we want to estimate yearly salary.

In [102]:
df["annual_salary"] = df["salary"] * 12
df["annual_salary"]

,annual_salary
0,60000
1,78000
2,72000
3,62400
4,84000


####Sometimes we create columns using division.

####For example, suppose we want to measure bonus percentage relative to salary.
####we can convert it into percentage form.

In [103]:
df['bonus_percentage'] = (df['bonus'] / df['salary']) * 100
df['bonus_percentage']

,bonus_percentage
0,10.000000
1,10.769231
2,10.000000
3,8.653846
4,11.428571


####Sometimes a column is created using a constant value. This means every row receives the same value.

In [104]:
df['country'] = "UK"
df['country']

,country
0,UK
1,UK
2,UK
3,UK
4,UK


####A new column can also be created by copying an existing column.

In [105]:
df['base_salary'] = df['salary']
df['base_salary']

,base_salary
0,5000
1,6500
2,6000
3,5200
4,7000


####Another common transformation is creating columns using comparisons.

####For example we may want to classify employees based on salary.

In [106]:
df['high_salary']= df['salary'] > 6000
df['high_salary']

,high_salary
0,False
1,True
2,False
3,False
4,True


####ou can also combine multiple operations when creating columns.

In [107]:
df['net_income'] = (df['salary'] + df['bonus']) - (df['salary'] * 0.10)
df['net_income']

,net_income
0,5000.0
1,6550.0
2,6000.0
3,5130.0
4,7100.0


#3. Apply() Function

####The apply() function allows you to apply a function to each element, row, or column in a DataFrame or Series. It is extremely flexible and can handle any custom operation, unlike simple vectorized arithmetic.
##1️ Applying a Function to a Single Column

####Suppose we want to increase all salaries by 10%. We can use apply() with a lambda function:

###When to Use apply()

####You should use apply() when:

####You need custom logic that can’t be done with simple arithmetic or vectorized operations

In [108]:
df['salary_increased'] = df['salary'].apply(lambda x: x* 1.10)
df['salary_increased']

,salary_increased
0,5500.0
1,7150.0
2,6600.0
3,5720.0
4,7700.0


##2️ Using apply() with a Named Function

####You can define a function first and pass it to apply().
####Each bonus is reduced by 10%.
####Named functions are useful when logic is complex or reused.

In [109]:
def bonus_tax(bonus):
  return bonus * 0.9
df['bonus_after_tax'] = df['bonus'].apply(bonus_tax)
df['bonus_after_tax']

,bonus_after_tax
0,450.0
1,630.0
2,540.0
3,405.0
4,720.0


#Rule of Thumb

####Use vectorized column operations if you can write the formula directly using +, -, *, /, etc.

####Use apply() only if your calculation involves complex logic, conditions, or functions that cannot be written as a simple column formula.

####For example:

###Simple formula (vectorized):

In [110]:
df["net_income"] = (df['salary'] + df['bonus']) - (df['salary'] * 0.10)

###Complex logic (need apply):

In [111]:
df["performance"] = df.apply(lambda row: "Excellent" if (row['salary']*0.3 + row['bonus']*0.5 + row['experience']*100)>5000 else "Good", axis=1)
df['performance']

,performance
0,Good
1,Good
2,Good
3,Good
4,Good


##4️ Using apply() for Conditional Transformation

####apply() allows if-else conditions to create new columns.

####Example: classify employees as “High” or “Low” salary:

In [112]:
df['salary_level'] = df['salary'].apply(lambda x: "high" if x > 6000 else "Low")
df['salary_level']

,salary_level
0,Low
1,high
2,Low
3,Low
4,high


###Apply with More Complex Functions

####You can create any custom logic, combining multiple operations.

####Example: assign a performance rating using salary, bonus, and experience:

In [113]:
def performance(row):
    score = row["salary"] * 0.3 + row["bonus"] * 0.5 + row["experience"] * 100
    if score > 5000:
        return "Excellent"
    elif score > 4000:
        return "Good"
    else:
        return "Average"
df["performance_rating"] = df.apply(performance, axis=1)
df["performance_rating"]

,performance_rating
0,Average
1,Average
2,Average
3,Average
4,Average


#Pandas map() Function

####The map() function is used on a Pandas Series (single column). It allows you to apply a function, dictionary, or mapping to transform the column values.

####It is similar to apply() but only works on a single column, and is usually faster and cleaner for simple transformations.
##What is map() in Pandas?

map() is a Series method — it works on single columns.

It allows you to transform each value of a column using:

**A function**

**A dictionary mapping**

**A Series mapping**

###Function:
It maps each value to a new value according to your rules.

###Why we use it:

To encode categorical variables

To replace or transform values in a column

To apply a function element-wise on a column

Faster and cleaner than apply() when only one column is involved

#1️ Using map() with a Function

####You can pass a function to map() to transform each value of the column.

####Example: add 500 bonus to every salary:

In [114]:
df['add_bonus'] = df['bonus'].map(lambda x: x+500)
df['add_bonus']

,add_bonus
0,1000
1,1200
2,1100
3,950
4,1300


##2 Using map() with a Dictionary (Most Common Use)

###map() shines when you want to map categorical values to numeric codes or replace values.

####Example: map cities to codes:

In [115]:
df['city']

,city
0,London
1,Paris
2,London
3,Berlin
4,Paris


###This is extremely useful for encoding categorical features for analysis or ML.

In [116]:
city_mapping = {"London": 1, "Paris": 2, "New York": 3}
df['city_codes'] = df['city'].map(city_mapping)
df['city_codes']

,city_codes
0,1.0
1,2.0
2,1.0
3,NaN
4,2.0


"London" → 1

"Paris" → 2

"New York" → 3

Any value not in the dictionary → NaN


##3 Using map() with a Series of Values

####You can also pass a Series or list to map values:

In [117]:
department_mapping = pd.Series({"Sales": 10, "Marketing": 20, "HR": 30})
df['department_code'] = df['department'].map(department_mapping)
df['department_code']

,department_code
0,10.0
1,NaN
2,NaN
3,10.0
4,NaN


##4 Using map() for String Operations

####You can also use map() for string transformations, but for most string operations, .str methods are preferred.

####Example:

In [118]:
df["name_upper"] = df["name"].map(lambda x: x.upper())
df['name_upper']

,name_upper
0,ALI KHAN
1,SARA AHMED
2,OMAR MALIK
3,AYESHA NOOR
4,BILAL RAZA


##5 Handling Missing Values with map()

####If some values are missing or not in the mapping dictionary, they will become NaN:

If a row has "New York" → result is NaN

You can fill it later with .fillna() if needed

In [119]:
df["city_code_dict"] = df["city"].map({"London":1, "Paris":2})
df['city_code_dict']

,city_code_dict
0,1.0
1,2.0
2,1.0
3,NaN
4,2.0


####fill null values with 0

In [120]:
df["city_code_dict"] = df["city"].map({"London":1, "Paris":2}).fillna(0)
df['city_code_dict']

,city_code_dict
0,1.0
1,2.0
2,1.0
3,0.0
4,2.0


#5. Lambda Functions

##1 What is a Lambda Function?

####A lambda function is an anonymous (no name) function in Python.

####It is short and lightweight, usually for simple operations.

###Syntax:

`lambda arguments: expression`

### Normal function
`def square(x):`
    `return x**2`

### Lambda function
`square = lambda x: x**2`

##2 Lambda in Pandas

####Lambda functions are commonly used with apply() and map() to transform DataFrame or Series values quickly without defining a separate function.

####x is each value of the salary column

Adds 500 to each salary

In [121]:
df['salary_plus_bonus'] = df['bonus'].apply(lambda x: x + 500)
df['salary_plus_bonus']

,salary_plus_bonus
0,1000
1,1200
2,1100
3,950
4,1300


####Example 2: Using lambda with apply(axis=1) on multiple columns

In [122]:
df["net_income"] = df.apply(lambda row: (row['salary'] + row['bonus']) - (row['salary']*0.10),axis=1)
df['net_income']

,net_income
0,5000.0
1,6550.0
2,6000.0
3,5130.0
4,7100.0


###Example 3: Using lambda with map()
Maps city names to codes

Any city not in the dictionary → 0

In [123]:
city_mapping = {"London":1, "Paris":2, "New York":3}
df['city_code'] = df['city'].map(lambda x: city_mapping.get(x,0))
df['city_code']

,city_code
0,1
1,2
2,1
3,0
4,2


#6. Renaming Columns in Pandas
###1️⃣ Why Rename Columns?

####To make column names more meaningful or readable

####To standardize column names for analysis or ML

####To match naming conventions (e.g., lowercase, no spaces, underscores)
###The Pandas rename() function allows you to rename columns by passing a dictionary.
####inplace=True → changes are applied to the original DataFrame

####inplace=False (default) → returns a new DataFrame

In [124]:
df.rename(columns = {'salary':'base_salary', 'bonus':'bonus_amount'},inplace = True)

##3 Rename All Columns at Once

####You can also rename all columns using df.columns:

`df.columns = ["employee_id", "name", "city", "department", "base_salary", "bonus_amount", "experience"]`
####Make sure the length matches the number of columns

####Useful for cleaning messy datasets from CSV or Excel

##4 Using Functions to Rename Columns

####You can pass a function to rename() to transform all column names:
####str.lower → converts all column names to lowercase

## Why str.lower does not have ()
`df.rename(columns=str.lower, inplace=True)`

####str.lower is a function object, not a function call.

####We are passing the function itself to rename().

####Pandas will call it internally on each column name.

####If you write str.lower() → it executes immediately and returns a string ('') — not a function — which breaks rename().

In [125]:
df.rename(columns =str.lower, inplace = True)
df

,employee_id,name,city,department,base_salary,bonus_amount,experience,total_salary,salary_after_tax,annual_salary,...,performance,salary_level,performance_rating,add_bonus,city_codes,department_code,name_upper,city_code_dict,salary_plus_bonus,city_code
0,101,Ali Khan,London,Sales,5000,500,3,5500,4500.0,60000,...,Good,Low,Average,1000,1.0,10.0,ALI KHAN,1.0,1000,1
1,102,Sara Ahmed,Paris,IT,6500,700,5,7200,5850.0,78000,...,Good,high,Average,1200,2.0,NaN,SARA AHMED,2.0,1200,2
2,103,Omar Malik,London,Finance,6000,600,4,6600,5400.0,72000,...,Good,Low,Average,1100,1.0,NaN,OMAR MALIK,1.0,1100,1
3,104,Ayesha Noor,Berlin,Sales,5200,450,2,5650,4680.0,62400,...,Good,Low,Average,950,NaN,10.0,AYESHA NOOR,0.0,950,0
4,105,Bilal Raza,Paris,IT,7000,800,6,7800,6300.0,84000,...,Good,high,Average,1300,2.0,NaN,BILAL RAZA,2.0,1300,2


####Lambda function → custom transformation for all columns
####Why lambda x: x.replace(" ", "_") does not have str in front
`df.rename(columns=lambda x: x.replace(" ", "_"), inplace=True)`

####x is each column name, which is already a string.

####So we can directly call replace() on it: x.replace(" ", "_")

####str.replace(x, " ", "_") is unnecessary because x is already a string instance.

In [126]:
df.rename(columns = lambda x: x.replace(" ", "_"),inplace = True)
df

,employee_id,name,city,department,base_salary,bonus_amount,experience,total_salary,salary_after_tax,annual_salary,...,performance,salary_level,performance_rating,add_bonus,city_codes,department_code,name_upper,city_code_dict,salary_plus_bonus,city_code
0,101,Ali Khan,London,Sales,5000,500,3,5500,4500.0,60000,...,Good,Low,Average,1000,1.0,10.0,ALI KHAN,1.0,1000,1
1,102,Sara Ahmed,Paris,IT,6500,700,5,7200,5850.0,78000,...,Good,high,Average,1200,2.0,NaN,SARA AHMED,2.0,1200,2
2,103,Omar Malik,London,Finance,6000,600,4,6600,5400.0,72000,...,Good,Low,Average,1100,1.0,NaN,OMAR MALIK,1.0,1100,1
3,104,Ayesha Noor,Berlin,Sales,5200,450,2,5650,4680.0,62400,...,Good,Low,Average,950,NaN,10.0,AYESHA NOOR,0.0,950,0
4,105,Bilal Raza,Paris,IT,7000,800,6,7800,6300.0,84000,...,Good,high,Average,1300,2.0,NaN,BILAL RAZA,2.0,1300,2


#7. Dropping Columns

##Dropping Columns in Pandas
###1 Why Drop Columns?

####Some columns may be irrelevant for analysis or ML

####Columns may be duplicated or redundant

####Helps reduce memory usage and make your dataset cleaner

####Makes your portfolio notebooks and reports professional

##2 How to Drop Columns Using drop()

####"bonus" → the column to drop

####axis=1 → tells Pandas we are dropping a column, not a row

####inplace=True → modifies the original DataFrame

####inplace=False (default) → returns a new DataFrame without changing the original

In [127]:
df.drop('annual_salary',axis=1,inplace = True)

##3. Dropping Multiple Columns

In [128]:
df.drop(df[['performance_rating',	'city_codes',	'department_code',	'name_upper',	'city_code_dict',	'city_code']],axis=1, inplace = True)

In [129]:
df.drop(["high_salary",	"net_income",	"salary_increased",	"bonus_after_tax",	"performance",	"salary_level",	"salary_plus_bonus"],axis =1, inplace=True)

##4 Drop Columns and Assign to a New DataFrame

####If you want to keep the original DataFrame:
####df remains unchanged

####df_new contains the dataset without the dropped columns
####We don't use inplace = True


In [130]:
df_new = df.drop(["salary_after_tax",	"bonus_percentage",	"country",	"base_salary",	"add_bonus"],axis=1)
df_new

,employee_id,name,city,department,bonus_amount,experience,total_salary
0,101,Ali Khan,London,Sales,500,3,5500
1,102,Sara Ahmed,Paris,IT,700,5,7200
2,103,Omar Malik,London,Finance,600,4,6600
3,104,Ayesha Noor,Berlin,Sales,450,2,5650
4,105,Bilal Raza,Paris,IT,800,6,7800


##Drops Columns using index numbers

In [131]:
df.drop(df.columns[[6, 7]],axis= 1,inplace=True)

#8. Assign Method in Pandas

####assign() is a DataFrame method to create new columns or modify existing columns.

####Returns a new DataFrame by default, leaving the original unchanged (unless you reassign it).

####Allows chainable operations, which makes your code cleaner and more professional.

##2 Basic Syntax
`df.assign(new_column = value_or_function)`

####new_column → name of the column to create

####value_or_function → can be:

####A constant value

####A function/lambda applied to the DataFrame

##3 Adding a Constant Column
####Adds a new column country with all values set to "USA"

In [132]:
df = df.assign(country = 'USA')
df

,employee_id,name,city,department,base_salary,bonus_amount,salary_after_tax,bonus_percentage,country,base_salary,add_bonus
0,101,Ali Khan,London,Sales,5000,500,4500.0,10.000000,USA,5000,1000
1,102,Sara Ahmed,Paris,IT,6500,700,5850.0,10.769231,USA,6500,1200
2,103,Omar Malik,London,Finance,6000,600,5400.0,10.000000,USA,6000,1100
3,104,Ayesha Noor,Berlin,Sales,5200,450,4680.0,8.653846,USA,5200,950
4,105,Bilal Raza,Paris,IT,7000,800,6300.0,11.428571,USA,7000,1300


##4 Adding Columns Using a Lambda Function

In [137]:
df = df.assign(net_income = lambda x: x.add_bonus + x.bonus_amount)

##5 Adding Multiple Columns at Once

In [139]:
df = df.assign(net_income = lambda x: x.add_bonus + x.bonus_amount,
               tax = lambda x: x.bonus_amount * 0.1)
df

,employee_id,name,city,department,base_salary,bonus_amount,salary_after_tax,bonus_percentage,country,base_salary,add_bonus,net_income,tax
0,101,Ali Khan,London,Sales,5000,500,4500.0,10.000000,USA,5000,1000,1500,50.0
1,102,Sara Ahmed,Paris,IT,6500,700,5850.0,10.769231,USA,6500,1200,1900,70.0
2,103,Omar Malik,London,Finance,6000,600,5400.0,10.000000,USA,6000,1100,1700,60.0
3,104,Ayesha Noor,Berlin,Sales,5200,450,4680.0,8.653846,USA,5200,950,1400,45.0
4,105,Bilal Raza,Paris,IT,7000,800,6300.0,11.428571,USA,7000,1300,2100,80.0


In [140]:
df.drop(df.columns[[9,10]],axis=1,inplace=True)

##6 Chain Operations Example
####Adds net_income

####Then creates a performance column based on net_income

####Chaining → professional, portfolio-ready code

In [145]:
import numpy as np

df = df.assign(net_income = lambda x: x.bonus_amount+ x.salary_after_tax) .assign(performance = lambda x: np.where(x.net_income > 5500, "Excellent", "Good" ))

#9. Replacing Values in Pandas Using replace()
##1 What is replace()?

####The replace() method allows you to replace specific values in a Series or DataFrame.

####Can handle single values, multiple values, or patterns (regex).

####Very useful for data cleaning, standardizing categories, or fixing errors.

##2 Basic Syntax
`df["column_name"].replace(old_value, new_value, inplace=False)`

####old_value → value you want to replace

####new_value → value to replace with

####inplace=True → changes the original DataFrame

####inplace=False (default) → returns a new DataFrame/Series

##3 Replacing a Single Value

In [151]:
df['city'].replace("London", "London City",inplace = False)

,city
0,London City
1,Paris
2,London City
3,Berlin
4,Paris


##4 Replacing Multiple Values

In [152]:
df['city'].replace({"London":"LDN", "Berlin": "BRN"},inplace = False)

,city
0,LDN
1,Paris
2,LDN
3,BRN
4,Paris


##5 Replace Across Entire DataFrame

In [154]:
df.replace({"bonus_amount": {500:600, 700:800}}, inplace = False)

,employee_id,name,city,department,bonus_amount,salary_after_tax,bonus_percentage,country,net_income,tax,performance
0,101,Ali Khan,London,Sales,600,4500.0,10.000000,USA,5000.0,50.0,Good
1,102,Sara Ahmed,Paris,IT,800,5850.0,10.769231,USA,6550.0,70.0,Excellent
2,103,Omar Malik,London,Finance,600,5400.0,10.000000,USA,6000.0,60.0,Excellent
3,104,Ayesha Noor,Berlin,Sales,450,4680.0,8.653846,USA,5130.0,45.0,Good
4,105,Bilal Raza,Paris,IT,800,6300.0,11.428571,USA,7100.0,80.0,Excellent


####Can also replace in multiple columns at once:

In [155]:
df.replace({"salary_after_tax": {4500:4600, 5400:5500}, "net_income": {5000:6000, 5130:5200}},inplace = False)

,employee_id,name,city,department,bonus_amount,salary_after_tax,bonus_percentage,country,net_income,tax,performance
0,101,Ali Khan,London,Sales,500,4600.0,10.000000,USA,6000.0,50.0,Good
1,102,Sara Ahmed,Paris,IT,700,5850.0,10.769231,USA,6550.0,70.0,Excellent
2,103,Omar Malik,London,Finance,600,5500.0,10.000000,USA,6000.0,60.0,Excellent
3,104,Ayesha Noor,Berlin,Sales,450,4680.0,8.653846,USA,5200.0,45.0,Good
4,105,Bilal Raza,Paris,IT,800,6300.0,11.428571,USA,7100.0,80.0,Excellent


##6 Using Regular Expressions (Regex) in Replace

####Very useful for string cleaning, e.g., removing extra spaces or symbols

In [156]:
df['name'].replace(r"\s+", inplace= False, regex=True)

,name
0,Ali Khan
1,Sara Ahmed
2,Omar Malik
3,Ayesha Noor
4,Bilal Raza


#10. Conditional Column Creation
##1 What is Conditional Column Creation?

####It means creating a new column whose values depend on a condition.

###Example idea:

salary	performance

6000	Excellent

4000	Good

####The value of performance depends on salary.

##2. Method 1 — Using np.where() (Most Common)

####This is the most commonly used method in data analysis.

####Syntax
`np.where(condition, value_if_true, value_if_false)`

In [160]:
df['performance'] = np.where(df['net_income']>5000, "Excellent", "Good")

In [161]:
df

,employee_id,name,city,department,bonus_amount,salary_after_tax,bonus_percentage,country,net_income,tax,performance
0,101,Ali Khan,London,Sales,500,4500.0,10.000000,USA,5000.0,50.0,Good
1,102,Sara Ahmed,Paris,IT,700,5850.0,10.769231,USA,6550.0,70.0,Excellent
2,103,Omar Malik,London,Finance,600,5400.0,10.000000,USA,6000.0,60.0,Excellent
3,104,Ayesha Noor,Berlin,Sales,450,4680.0,8.653846,USA,5130.0,45.0,Excellent
4,105,Bilal Raza,Paris,IT,800,6300.0,11.428571,USA,7100.0,80.0,Excellent


##3 Method 2 — Multiple Conditions using np.select()

####When you have more than one condition, use np.select().

###Syntax
`np.select([condition1, condition2], [value1, value2], default=value)`

In [165]:
conditions = [df['net_income']<6000, df['net_income']>6000]
values = ['Excellent', 'Good']
df['Level'] = np.select(conditions, values, default='low')
df['Level']

,Level
0,Excellent
1,Good
2,low
3,Excellent
4,Good


##4 Method 3 — Using apply() with lambda

####You can also create conditional columns using apply.

In [166]:
df['performance'] = df.apply(lambda row: "Excellent" if row['net_income']> 6000 else 'Good', axis = 1)
df['performance']

,performance
0,Good
1,Excellent
2,Good
3,Good
4,Excellent
